### HALI v3 — HPV Awareness & Learning Initiative
#### LangGraph | Week 4 Community Contribution

A persistent CHV (Community Health Volunteer) support tool built with LangGraph.

**What's new vs Week 2/3:**
- **Persistent memory** — conversations saved to SQLite, CHV picks up where they left off
- **Graph routing** — messages routed to specialist nodes based on content

**Nodes:**
- `router` — classifies the message using structured output
- `myth_buster` — handles parent objections with evidence
- `eligibility` — answers age/dose/access questions  
- `follow_up` — helps log hesitant families for follow-up
- `general` — everything else about HPV and cervical cancer

**Context:** Kenya has 3,400 cervical cancer deaths per year. 
CHVs are the frontline — this tool helps them respond confidently in the field.


In [ ]:
# Imports and setup
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from pydantic import BaseModel
from typing import Annotated
from typing_extensions import TypedDict
import operator

load_dotenv(override=True)
llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
# Define the graph state
class HaliState(TypedDict):
    messages: Annotated[list, operator.add]
    chv_name: str
    county: str
    families_helped: int
    last_myth: str


In [ ]:
# Router — decides which node handles the message
class Route(BaseModel):
    destination: str  # "myth_buster" | "eligibility" | "follow_up" | "general"
    reasoning: str

router_llm = llm.with_structured_output(Route)

def router_node(state: HaliState):
    last_message = state["messages"][-1].content
    route = router_llm.invoke([
        SystemMessage(content="""You are a router for HALI, an HPV vaccine support tool for Community Health Volunteers in Kenya.
Route the message to one of:
- myth_buster: if the CHV is dealing with a vaccine myth or parent objection
- eligibility: if they are asking about age, dose, or eligibility
- follow_up: if they want to log or check on a family
- general: anything else about HPV or cervical cancer"""),
        HumanMessage(content=last_message)
    ])
    return {"messages": [], "last_myth": last_message if route.destination == "myth_buster" else state.get("last_myth", "")}

def route_edge(state: HaliState):
    last_message = state["messages"][-1].content
    route = router_llm.invoke([
        SystemMessage(content="""Route to: myth_buster, eligibility, follow_up, or general."""),
        HumanMessage(content=last_message)
    ])
    return route.destination


In [ ]:
# Specialist nodes
def myth_buster_node(state: HaliState):
    response = llm.invoke([
        SystemMessage(content="""You support Community Health Volunteers (CHVs) in Kenya dealing with HPV vaccine myths.
Give a short, evidence-based response they can use immediately in the field.
Cite WHO or Kenya MOH where relevant. Be practical, not academic."""),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}

def eligibility_node(state: HaliState):
    response = llm.invoke([
        SystemMessage(content="""You answer eligibility questions about the HPV vaccine in Kenya.
Key facts: girls 9-14 are the primary target, one dose is enough, the vaccine is free at public facilities.
Girls up to 26 can still benefit. Boys are not yet in the Kenya programme."""),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}

def follow_up_node(state: HaliState):
    families = state.get("families_helped", 0) + 1
    response = llm.invoke([
        SystemMessage(content="""You help CHVs log and track hesitant families for follow-up.
Acknowledge what they've shared, suggest a follow-up approach, and encourage them."""),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)], "families_helped": families}

def general_node(state: HaliState):
    response = llm.invoke([
        SystemMessage(content="""You are HALI, an HPV vaccine companion for Community Health Volunteers in Kenya.
Answer general questions about HPV, cervical cancer, and the vaccine programme warmly and clearly."""),
        *state["messages"]
    ])
    return {"messages": [AIMessage(content=response.content)]}


In [ ]:
# Build the graph with persistent memory
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

conn = sqlite3.connect("hali_memory.db", check_same_thread=False)
memory = SqliteSaver(conn)

graph = StateGraph(HaliState)

graph.add_node("router", router_node)
graph.add_node("myth_buster", myth_buster_node)
graph.add_node("eligibility", eligibility_node)
graph.add_node("follow_up", follow_up_node)
graph.add_node("general", general_node)

graph.set_entry_point("router")

graph.add_conditional_edges("router", route_edge, {
    "myth_buster": "myth_buster",
    "eligibility": "eligibility",
    "follow_up": "follow_up",
    "general": "general",
})

graph.add_edge("myth_buster", END)
graph.add_edge("eligibility", END)
graph.add_edge("follow_up", END)
graph.add_edge("general", END)

app = graph.compile(checkpointer=memory)


In [ ]:
# Test the graph
config = {"configurable": {"thread_id": "chv-amina-wajir"}}

def chat(message):
    result = app.invoke(
        {"messages": [HumanMessage(content=message)], "chv_name": "Amina", "county": "Wajir", "families_helped": 0, "last_myth": ""},
        config=config
    )
    print(result["messages"][-1].content)

chat("A mother says the vaccine will make her daughter infertile. What do I tell her?")


In [ ]:
# Test eligibility routing — same thread so it remembers context
chat("My CHV colleague says a 16-year-old missed the school programme. Can she still get vaccinated?")


In [ ]:
# Test follow-up routing
chat("I spoke to a hesitant family in Wajir today — the mother wants more time to think. How do I log this?")


In [ ]:
# Gradio UI
import gradio as gr

def respond(message, history):
    result = app.invoke(
        {"messages": [HumanMessage(content=message)], 
         "chv_name": "", "county": "", "families_helped": 0, "last_myth": ""},
        config=config
    )
    return result["messages"][-1].content

gr.ChatInterface(
    fn=respond,
    type="messages",
    title="HALI — CHV Support Tool",
    description="Support for Community Health Volunteers in Kenya",
    examples=[
        "A mother says the vaccine will make her daughter infertile. What do I tell her?",
        "Can a 16-year-old still get the vaccine?",
        "I visited a hesitant family today — how do I log this for follow-up?",
    ],
).launch()
